In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,RobustScaler, MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
import datetime
import re
from sklearn.metrics import mean_absolute_error
import os

#physical_devices = tf.config.list_physical_devices('GPU')
#if physical_devices:
#    print(f"NVIDIA GPU erkannt: {len(physical_devices)} GPU(s) verfügbar.")
#    try:
#        for gpu in physical_devices:
#            tf.config.experimental.set_memory_growth(gpu, True)
#        print("GPU Memory Growth aktiviert (verhindert Out-of-Memory Fehler).")
#    except RuntimeError as e:
#        print(e)
#else:
#    print("Keine GPU erkannt. Training wird auf der CPU ausgeführt.")

df = pd.read_csv('training_data_ready.csv')
#df = df.drop([21,43,92,159], axis=0) #[21,43,92,159] # [92,99,31,169,150,43] #[17,21,43,53,62,92,129,150,159,166,169] #[15,17,21,30,35,43,53,62,67,72,73,75,91,92,129,139,142,150,159,166,169]
target_cols = ['x', 'y', 'z','Gewicht'] 
x_cols = [col for col in df.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit','Gewicht']]
    
X = df[x_cols]
y = df[target_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)



In [ ]:
def create_neural_network(X_train, y_train):

    # 5. Architektur des Neuronalen Netzes definieren
    model = models.Sequential([
        # Eingabeschicht (Größe entspricht Anzahl der Features, z.B. 28)
        layers.Input(shape=(X_train.shape[1],)),

        layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.1)),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1),
         # 1 versteckte Schicht
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.1)),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1), # Verhindert Overfitting

        # 2 versteckte Schicht
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.1)),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1), # Verhindert Overfitting
        
        # 3 versteckte Schicht
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.1)),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1), # Verhindert Overfitting
        
        # Ausgabeschicht: 4 Neuronen für x, y, z, Gewicht (keine Aktivierungsfunktion für Regression)
        layers.Dense(4)
    ])
    
    # 6. Kompilieren
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # 7. Training
    print("Starte Training...")

    # 3. Early Stopping
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=500, 
        restore_best_weights=True,
        verbose=0
    )
    history = model.fit(
        X_train_scaled, y_train,
        epochs=5000,
        batch_size=8,
        validation_data=(X_val_scaled, y_val),
        callbacks=[early_stopping],
        verbose=0
    )

        # 5. Besten Validierungsfehler auslesen
    min_val_loss = min(history.history['val_loss'])
    corresponding_mae = history.history['val_mae'][history.history['val_loss'].index(min_val_loss)]
    epochs_trained = len(history.history['val_loss'])
    
    print(f" -> Fertig nach {epochs_trained} Epochen. Bester Val-Loss: {min_val_loss:.4f} mit MAE: {corresponding_mae:.4f}")
    
    return model

In [ ]:
model = create_neural_network(X_train_scaled, y_train)

Starte Training...


In [ ]:
import joblib
joblib.dump(scaler, 'robust_scaler_Model_with_Weights_robust_[512, 256, 128, 64]_L2_0.1_Drop_0.1_lr_0.001.pkl')
#Model Save:
model.save('Model_with_Weights_robust_[512, 256, 128, 64]_L2_0.1_Drop_0.1_lr_0.001.keras')

In [ ]:
y_pred = model.predict(X_test_scaled)

y_pred_df = pd.DataFrame(
    y_pred, columns=["x_pred", "y_pred", "z_pred","Gewicht_pred"], index=y_test.index
)

results = y_test.copy()
results[["x_pred", "y_pred", "z_pred","Gewicht_pred"]] = y_pred_df

results["abs_error_x"] = np.abs(results["x_pred"] - results["x"])
results["abs_error_y"] = np.abs(results["y_pred"] - results["y"])
results["abs_error_z"] = np.abs(results["z_pred"] - results["z"])
results["abs_error_Gewicht"] = np.abs(results["Gewicht_pred"] - results["Gewicht"])

results["mse_x"] = (results["x_pred"] - results["x"]) ** 2
results["mse_y"] = (results["y_pred"] - results["y"]) ** 2
results["mse_z"] = (results["z_pred"] - results["z"]) ** 2
results["mse_Gewicht"] = (results["Gewicht_pred"] - results["Gewicht"]) ** 2

#restliche_spalten = df.loc[y_test.index].drop(
#    columns=["x", "y", "z"], errors="ignore"
#)
#doppelte_spalten = [c for c in restliche_spalten.columns if c in results.columns]
#restliche_spalten = restliche_spalten.drop(columns=doppelte_spalten)

results = pd.concat([results, df.loc[y_test.index, ["source_file"]]], axis=1)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = model.__class__.__name__

try:
    dense_units = [
        str(layer.units) for layer in model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)
    opt_name = model.optimizer.__class__.__name__
    try:
        lr = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"
    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    param_str = "keras_nn"

filename = f"Auswertung_test_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)

#results.to_csv(filename, index=True, sep=";", decimal=",")

print("Auswertung Testdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

print(f"\n[INFO] Datei erfolgreich gespeichert unter: {filename}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Auswertung Testdatensatz:
  - MAE x: 11.9302
  - MAE y: 14.1442
  - MAE z: 3.8931
  - MAE Gewicht: 6.0940
  - MSE x: 343.8997
  - MSE y: 510.8649
  - MSE z: 45.0633
  - MSE Gewicht: 97.8808

[INFO] Datei erfolgreich gespeichert unter: Auswertung_test_Sequential_Arch_512-256-128-64-4_Adam_lr_0.0010000000474974513_20260617_171839.csv


In [ ]:
results.style.format(
    {
        "x": "{:.2f}",
        "x_pred": "{:.2f}",
        "abs_error_x": "{:.2f}",
        "y": "{:.2f}",
        "y_pred": "{:.2f}",
        "abs_error_y": "{:.2f}",
        "z": "{:.2f}",
        "z_pred": "{:.2f}",
        "abs_error_z": "{:.2f}",
        "Gewicht": "{:.2f}",
        "Gewicht_pred": "{:.2f}",
        "abs_error_Gewicht": "{:.2f}",
    },
    decimal=",",
).background_gradient(
    subset=["abs_error_x", "abs_error_y", "abs_error_z","abs_error_Gewicht"], cmap="Reds"
)

,x,y,z,Gewicht,x_pred,y_pred,z_pred,Gewicht_pred,abs_error_x,abs_error_y,abs_error_z,abs_error_Gewicht,mse_x,mse_y,mse_z,mse_Gewicht,source_file
103,"20,00","87,50","47,50","51,70","33,85","52,74","56,14","49,98","13,85","34,76","8,64","1,72","191,830277","1208,230168","74,713214","2,966937",Lumi_V_L_L_6.csv
139,"81,39","46,35","20,00","58,10","66,69","39,74","23,64","61,73","14,70","6,61","3,64","3,63","216,029370","43,733087","13,253928","13,161448",Endboss_L_U_S_X_3.csv
80,"47,50","87,50","20,00","51,70","55,39","60,55","18,90","47,16","7,89","26,95","1,10","4,54","62,253294","726,307270","1,207522","20,598649",Lumi_L_L_8.csv
58,"25,00","50,00","31,75","127,80","48,25","25,92","40,37","127,24","23,25","24,08","8,62","0,56","540,368105","580,042438","74,331545","0,314055",Rechteck_V_L_L_2.csv
100,"20,00","87,50","47,50","51,70","47,35","74,98","24,10","52,96","27,35","12,52","23,40","1,26","748,009062","156,657293","547,400658","1,599560",Lumi_V_L_L_3.csv
30,"53,33","91,00","30,00","443,80","92,87","56,51","30,29","442,64","39,54","34,49","0,29","1,16","1563,560844","1189,291567","0,085293","1,350810",3eck_L_L_4.csv
107,"47,50","32,50","17,25","19,30","41,78","37,31","11,25","21,69","5,72","4,81","6,00","2,39","32,731857","23,098367","35,956674","5,735329",Durch_L_U_D_O_4.csv
84,"47,50","87,50","20,00","51,70","52,05","75,11","21,03","52,06","4,55","12,39","1,03","0,36","20,734616","153,635942","1,058102","0,132018",Lumi_L_L_12.csv
166,"12,50","120,00","17,50","71,60","70,20","28,14","19,17","109,54","57,70","91,86","1,67","37,94","3329,815287","8437,985687","2,793980","1439,704774",Rechteck_Lang_L_L_3.csv
111,"47,50","32,50","5,75","19,30","43,17","38,68","7,88","21,33","4,33","6,18","2,13","2,03","18,727713","38,239332","4,519711","4,104028",Durch_L_U_D_U_2.csv


In [ ]:
y_pred = model.predict(X_train_scaled)

y_pred_df = pd.DataFrame(
    y_pred, columns=["x_pred", "y_pred", "z_pred", "Gewicht_pred"], index=y_train.index
)

results = y_train.copy()
results[["x_pred", "y_pred", "z_pred", "Gewicht_pred"]] = y_pred_df

results["abs_error_x"] = np.abs(results["x_pred"] - results["x"])
results["abs_error_y"] = np.abs(results["y_pred"] - results["y"])
results["abs_error_z"] = np.abs(results["z_pred"] - results["z"])
results["abs_error_Gewicht"] = np.abs(results["Gewicht_pred"] - results["Gewicht"])

results["mse_x"] = (results["x_pred"] - results["x"]) ** 2
results["mse_y"] = (results["y_pred"] - results["y"]) ** 2
results["mse_z"] = (results["z_pred"] - results["z"]) ** 2
results["mse_Gewicht"] = (results["Gewicht_pred"] - results["Gewicht"]) ** 2

##restliche_spalten = df.loc[y_train.index].drop(
#    columns=["x", "y", "z"], errors="ignore"
#)
#doppelte_spalten = [c for c in restliche_spalten.columns if c in results.columns]
#restliche_spalten = restliche_spalten.drop(columns=doppelte_spalten)

#results = pd.concat([results, restliche_spalten], axis=1)
results = pd.concat([results, df.loc[y_train.index, ["source_file"]]], axis=1)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = model.__class__.__name__

try:
    dense_units = [
        str(layer.units) for layer in model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)
    opt_name = model.optimizer.__class__.__name__
    try:
        lr = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"
    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    param_str = "keras_nn"

filename = f"Auswertung_train_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)

#results.to_csv(filename, index=True, sep=";", decimal=",")

print("Auswertung Trainingsdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

print(f"\n[INFO] Datei erfolgreich gespeichert unter: {filename}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
Auswertung Trainingsdatensatz:
  - MAE x: 2.3025
  - MAE y: 2.0184
  - MAE z: 1.5001
  - MAE Gewicht: 2.2128
  - MSE x: 11.8852
  - MSE y: 16.0710
  - MSE z: 5.6156
  - MSE Gewicht: 10.7581

[INFO] Datei erfolgreich gespeichert unter: Auswertung_train_Sequential_Arch_512-256-128-64-4_Adam_lr_0.0010000000474974513_20260617_171840.csv


In [ ]:
results.style.format(
    {
        "x": "{:.2f}",
        "x_pred": "{:.2f}",
        "abs_error_x": "{:.2f}",
        "y": "{:.2f}",
        "y_pred": "{:.2f}",
        "abs_error_y": "{:.2f}",
        "z": "{:.2f}",
        "z_pred": "{:.2f}",
        "abs_error_z": "{:.2f}",
        "Gewicht": "{:.2f}",
        "Gewicht_pred": "{:.2f}",
        "abs_error_Gewicht": "{:.2f}",
    },
    decimal=",",
).background_gradient(
    subset=["abs_error_x", "abs_error_y", "abs_error_z","abs_error_Gewicht"], cmap="Reds"
)

,x,y,z,Gewicht,x_pred,y_pred,z_pred,Gewicht_pred,abs_error_x,abs_error_y,abs_error_z,abs_error_Gewicht,mse_x,mse_y,mse_z,mse_Gewicht,source_file
161,"120,00","12,50","17,50","71,60","121,85","14,11","18,94","73,21","1,85","1,61","1,44","1,61","3,409747","2,606555","2,059798","2,590893",Rechteck_Lang_L_U_4.csv
2,"50,00","50,00","50,00","185,50","49,52","50,25","48,11","185,76","0,48","0,25","1,89","0,26","0,234530","0,061800","3,577218","0,066648",4eck_g_O_V_u_3.csv
121,"54,50","47,50","20,00","43,90","50,47","48,38","19,67","43,96","4,03","0,88","0,33","0,06","16,265435","0,778321","0,112012","0,003102",6eck_L_U_6.csv
115,"47,50","32,50","5,75","19,30","46,83","33,27","8,00","20,70","0,67","0,77","2,25","1,40","0,442581","0,595223","5,081486","1,963128",Durch_L_U_D_U_6.csv
70,"87,50","47,50","20,00","51,70","85,19","47,18","19,63","51,09","2,31","0,32","0,37","0,61","5,335349","0,101032","0,135890","0,369044",Lumi_L_U_11.csv
136,"64,61","46,35","20,00","58,10","62,24","44,61","21,39","55,81","2,37","1,74","1,39","2,29","5,599565","3,023325","1,918987","5,229498",Endboss_L_U_S_0_3.csv
47,"31,75","50,00","25,00","127,80","31,04","49,08","25,54","130,05","0,71","0,92","0,54","2,25","0,499572","0,848419","0,295752","5,060317",Rechteck_L_L_7.csv
27,"53,33","91,00","30,00","443,80","56,83","89,44","31,95","451,87","3,50","1,56","1,95","8,07","12,228126","2,443552","3,792839","65,056374",3eck_L_L_1.csv
145,"46,35","81,39","20,00","58,10","44,60","81,00","20,47","56,57","1,75","0,39","0,47","1,53","3,077263","0,151731","0,222950","2,352027",Endboss_L_L_S_Y_3.csv
86,"47,50","20,00","87,50","51,70","48,13","20,45","85,97","52,89","0,63","0,45","1,53","1,19","0,397445","0,200106","2,348724","1,414447",Lumi_H_L_U_1.csv


In [ ]:
os.makedirs("Modelle2", exist_ok=True)

drop_variants = {
    "No Drop": [],
    "Variant 1": [21, 43, 92, 159],
    "Variant 2": [92, 99, 31, 169, 150, 43],
    #"Variant 3": [17, 21, 43, 53, 62, 92, 129, 150, 159, 166, 169],
    #"Variant 4": [15, 17, 21, 30, 35, 43, 53, 62, 67, 72, 73, 75, 91, 92, 129, 139, 142, 150, 159, 166, 169]
}

column_variants = {
    "All Features": lambda cols: cols,
    "Without A_1": lambda cols: [c for c in cols if not c.startswith('A_1_')]
}

model_configs = [
    {"name": "Model 1", "scaler": "robust", "layers": [512, 256, 128, 64], "l2": 0.0, "dropout": 0.1, "lr": 0.005},
    {"name": "Model 2", "scaler": "robust", "layers": [128, 64, 32, 16], "l2": 0.0, "dropout": 0.1, "lr": 0.01},
    {"name": "Model 3", "scaler": "robust", "layers": [128, 64, 32], "l2": 0.1, "dropout": 0.1, "lr": 0.01},
    {"name": "Model 4", "scaler": "robust", "layers": [128, 64, 32, 16], "l2": 0.1, "dropout": 0.1, "lr": 0.01},
    {"name": "Model 5", "scaler": "robust", "layers": [256, 128, 64], "l2": 0.2, "dropout": 0.0, "lr": 0.01}
]

all_results = []
target_cols = ['x', 'y', 'z', 'Gewicht']

for drop_name, drop_indices in drop_variants.items():
    for col_var_name, col_filter_fn in column_variants.items():
        df_full = pd.read_csv('training_data_ready.csv')
        indices_to_drop = [i for i in drop_indices if i in df_full.index]
        df_exp = df_full.drop(indices_to_drop, axis=0)

        base_x_cols = [col for col in df_exp.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit']]
        
        x_cols = col_filter_fn(base_x_cols)

        X = df_exp[x_cols]
        y = df_exp[target_cols]

        X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=42)

        for config in model_configs:
            print(f"Starte {drop_name} | {col_var_name} mit {config['name']}...")

            scaler = RobustScaler()

            X_train_scaled = scaler.fit_transform(X_train)
            X_val_scaled = scaler.transform(X_val)
            X_test_scaled = scaler.transform(X_test)

            model = models.Sequential()
            model.add(layers.Input(shape=(X_train_scaled.shape[1],)))

            for units in config["layers"]:
                model.add(layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(config["l2"])))
                if config["dropout"] > 0:
                    model.add(layers.Dropout(config["dropout"]))

            model.add(layers.Dense(4))

            optimizer = tf.keras.optimizers.Adam(learning_rate=config["lr"])
            model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

            early_stopping = EarlyStopping(monitor='val_loss', patience=500, restore_best_weights=True, verbose=0)

            history = model.fit(
                X_train_scaled, y_train,
                epochs=5000,
                batch_size=8,
                validation_data=(X_val_scaled, y_val),
                callbacks=[early_stopping],
                verbose=0
            )

            min_val_loss = min(history.history['val_loss'])
            mae_key = 'val_mae' if 'val_mae' in history.history else 'val_mean_absolute_error'
            if mae_key in history.history:
                corresponding_mae = history.history[mae_key][history.history['val_loss'].index(min_val_loss)]
            else:
                corresponding_mae = None
            epochs_trained = len(history.history['val_loss'])

            print(f" -> Fertig nach {epochs_trained} Epochen. Bester Val-Loss: {min_val_loss:.4f}")

            layers_str = '-'.join(map(str, config['layers']))

            col_str = "noA1" if col_var_name == "Without A_1" else "all"
            drop_str = drop_name.replace(' ', '')
            base_name = f"{drop_str}_{col_str}_{config['scaler']}_layers_{layers_str}_l2_{config['l2']}_drop_{config['dropout']}_lr_{config['lr']}_valloss_{min_val_loss:.4f}"

            model_path = f"Modelle2/model_{base_name}.keras"
            model.save(model_path)

            scaler_path = f"Modelle2/scaler_{base_name}.pkl"
            joblib.dump(scaler, scaler_path)

            all_results.append({
                'drop_variant': drop_name,
                'col_variant': col_var_name,
                'scaler': config['scaler'],
                'layers': str(config['layers']),
                'l2': config['l2'],
                'dropout': config['dropout'],
                'lr': config['lr'],
                'val_loss': min_val_loss,
                'val_mae': corresponding_mae,
                'epochs': epochs_trained,
                'model_path': model_path,
                'scaler_path': scaler_path,
                'model_object': model
            })

results_df = pd.DataFrame(all_results)
display(results_df)

Starte No Drop | All Features mit Model 1...
 -> Fertig nach 2338 Epochen. Bester Val-Loss: 53.9717
Starte No Drop | All Features mit Model 2...
 -> Fertig nach 1572 Epochen. Bester Val-Loss: 65.9828
Starte No Drop | All Features mit Model 3...
 -> Fertig nach 1208 Epochen. Bester Val-Loss: 109.6117
Starte No Drop | All Features mit Model 4...
 -> Fertig nach 2100 Epochen. Bester Val-Loss: 94.2897
Starte No Drop | All Features mit Model 5...
 -> Fertig nach 1641 Epochen. Bester Val-Loss: 78.0956
Starte No Drop | Without A_1 mit Model 1...
 -> Fertig nach 1008 Epochen. Bester Val-Loss: 104.9204
Starte No Drop | Without A_1 mit Model 2...
 -> Fertig nach 1254 Epochen. Bester Val-Loss: 69.8856
Starte No Drop | Without A_1 mit Model 3...
 -> Fertig nach 1317 Epochen. Bester Val-Loss: 103.2037
Starte No Drop | Without A_1 mit Model 4...
 -> Fertig nach 1511 Epochen. Bester Val-Loss: 105.1995
Starte No Drop | Without A_1 mit Model 5...
 -> Fertig nach 1382 Epochen. Bester Val-Loss: 82.0381
S

,drop_variant,col_variant,scaler,layers,l2,dropout,lr,val_loss,val_mae,epochs,model_path,scaler_path,model_object
0,No Drop,All Features,robust,"[512, 256, 128, 64]",0.0,0.1,0.005,53.971748,5.370367,2338,Modelle2/model_NoDrop_all_robust_layers_512-25...,Modelle2/scaler_NoDrop_all_robust_layers_512-2...,<keras.engine.sequential.Sequential object at ...
1,No Drop,All Features,robust,"[128, 64, 32, 16]",0.0,0.1,0.010,65.982765,6.168232,1572,Modelle2/model_NoDrop_all_robust_layers_128-64...,Modelle2/scaler_NoDrop_all_robust_layers_128-6...,<keras.engine.sequential.Sequential object at ...
2,No Drop,All Features,robust,"[128, 64, 32]",0.1,0.1,0.010,109.611732,5.561422,1208,Modelle2/model_NoDrop_all_robust_layers_128-64...,Modelle2/scaler_NoDrop_all_robust_layers_128-6...,<keras.engine.sequential.Sequential object at ...
3,No Drop,All Features,robust,"[128, 64, 32, 16]",0.1,0.1,0.010,94.289658,5.497643,2100,Modelle2/model_NoDrop_all_robust_layers_128-64...,Modelle2/scaler_NoDrop_all_robust_layers_128-6...,<keras.engine.sequential.Sequential object at ...
4,No Drop,All Features,minmax,"[256, 128, 64]",0.2,0.0,0.010,78.095581,5.974323,1641,Modelle2/model_NoDrop_all_minmax_layers_256-12...,Modelle2/scaler_NoDrop_all_minmax_layers_256-1...,<keras.engine.sequential.Sequential object at ...
5,No Drop,Without A_1,robust,"[512, 256, 128, 64]",0.0,0.1,0.005,104.920410,8.078972,1008,Modelle2/model_NoDrop_noA1_robust_layers_512-2...,Modelle2/scaler_NoDrop_noA1_robust_layers_512-...,<keras.engine.sequential.Sequential object at ...
6,No Drop,Without A_1,robust,"[128, 64, 32, 16]",0.0,0.1,0.010,69.885551,6.721982,1254,Modelle2/model_NoDrop_noA1_robust_layers_128-6...,Modelle2/scaler_NoDrop_noA1_robust_layers_128-...,<keras.engine.sequential.Sequential object at ...
7,No Drop,Without A_1,robust,"[128, 64, 32]",0.1,0.1,0.010,103.203728,5.848161,1317,Modelle2/model_NoDrop_noA1_robust_layers_128-6...,Modelle2/scaler_NoDrop_noA1_robust_layers_128-...,<keras.engine.sequential.Sequential object at ...
8,No Drop,Without A_1,robust,"[128, 64, 32, 16]",0.1,0.1,0.010,105.199516,5.516380,1511,Modelle2/model_NoDrop_noA1_robust_layers_128-6...,Modelle2/scaler_NoDrop_noA1_robust_layers_128-...,<keras.engine.sequential.Sequential object at ...
9,No Drop,Without A_1,minmax,"[256, 128, 64]",0.2,0.0,0.010,82.038139,5.707717,1382,Modelle2/model_NoDrop_noA1_minmax_layers_256-1...,Modelle2/scaler_NoDrop_noA1_minmax_layers_256-...,<keras.engine.sequential.Sequential object at ...
